# Clase ETL — Etapa Load

En este notebook vamos a tomar datos ya **extraídos y transformados** y los vamos a cargar (load) en tres destinos distintos, representativos de situaciones reales:

1. **MongoDB Atlas** — base de datos en la nube (destino estructurado tipo NoSQL).
2. **Bucket de objetos en Cloudflare R2** — archivo Parquet en un *data lake*.
3. **Bucket de objetos en Cloudflare R2** — reporte PDF para compartir o archivar.

La idea es que vean que "cargar" no significa una sola cosa: el destino se elige según para qué se van a usar después los datos.


## 1. Instalación de dependencias

Corran esta celda una sola vez.

```bash
conda install pandas pymongo "pymongo[srv]" pyarrow s3fs boto3 python-dotenv
```

Qué hace cada uno:
- `pandas` — manejo de DataFrames
- `pymongo` (con extra `srv`) — cliente de MongoDB
- `pyarrow` — backend para leer/escribir Parquet
- `s3fs` + `boto3` — acceso al bucket R2 (que habla protocolo S3)


> Las credenciales viven en un archivo `.env` en la raíz del repo (está en `.gitignore`) y se cargan con `python-dotenv`. Nunca se escriben en el notebook ni se suben a git.

In [ ]:
from dotenv import load_dotenv, find_dotenv

# Busca el archivo .env subiendo desde la carpeta actual (lo encuentra en la raiz del repo)
# y carga sus variables al entorno. No pisa variables ya definidas en el shell.
load_dotenv(find_dotenv(usecwd=True))

import os

# MongoDB Atlas
MONGO_URI = os.environ["MONGO_URI"]
MONGO_DB = os.environ.get("MONGO_DB", "clase_etl")
MONGO_COLLECTION = os.environ.get("MONGO_COLLECTION", "consultas")

In [ ]:
# Cloudflare R2 (S3-compatible)
R2_ENDPOINT = os.environ["R2_ENDPOINT_URL"]  # sin el nombre del bucket al final
R2_ACCESS_KEY = os.environ["R2_ACCESS_KEY_ID"]
R2_SECRET_KEY = os.environ["R2_SECRET_ACCESS_KEY"]
R2_BUCKET = os.environ.get("R2_BUCKET", "diplo-ai")

Cambien la siguiente variable por su nombre:

In [ ]:
# Identificador del alumno (para que no se pisen entre sí)
# Ejemplo: "flavio-colavecchia". Usá guiones en lugar de espacios (y evitá tildes):
# el nombre se convierte en una "carpeta" dentro del bucket y, en la URL del objeto,
# cada espacio se codifica como %20 (flavio%20colavecchia), lo que complica los links.
ALUMNO = "flavio-colavecchia"  # cambiar por tu nombre/legajo

## 3. Datos de ejemplo

Simulamos una **lista de pacientes citados para el día**. Son datos ficticios que ya pasaron por las etapas E y T del ETL: ya tienen tipos consistentes, campos normalizados y una columna `alumno` para identificar el origen de la carga.


In [4]:
import pandas as pd
from datetime import datetime

fecha_turnos = "2026-05-23"

datos = [
    {"turno": "08:00", "paciente_id": "P001", "paciente": "Ana Ruiz", "edad": 45, "cobertura": "OSDE", "profesional": "Dra. Medina", "especialidad": "Clinica medica", "motivo": "control anual", "estado": "confirmado", "prioridad": "normal"},
    {"turno": "08:30", "paciente_id": "P002", "paciente": "Carlos Vega", "edad": 62, "cobertura": "PAMI", "profesional": "Dr. Salas", "especialidad": "Cardiologia", "motivo": "dolor toracico", "estado": "confirmado", "prioridad": "alta"},
    {"turno": "09:00", "paciente_id": "P003", "paciente": "Lucia Moreno", "edad": 38, "cobertura": "Swiss Medical", "profesional": "Dra. Medina", "especialidad": "Clinica medica", "motivo": "chequeo laboral", "estado": "confirmado", "prioridad": "normal"},
    {"turno": "09:30", "paciente_id": "P004", "paciente": "Jorge Alvarez", "edad": 71, "cobertura": "PAMI", "profesional": "Dr. Salas", "especialidad": "Cardiologia", "motivo": "seguimiento hipertension", "estado": "confirmado", "prioridad": "alta"},
    {"turno": "10:00", "paciente_id": "P005", "paciente": "Marta Perez", "edad": 29, "cobertura": "Particular", "profesional": "Dra. Ibarra", "especialidad": "Dermatologia", "motivo": "control de lunares", "estado": "pendiente", "prioridad": "normal"},
    {"turno": "10:30", "paciente_id": "P006", "paciente": "Nicolas Ferrer", "edad": 52, "cobertura": "Galeno", "profesional": "Dra. Medina", "especialidad": "Clinica medica", "motivo": "laboratorio alterado", "estado": "confirmado", "prioridad": "media"},
    {"turno": "11:00", "paciente_id": "P007", "paciente": "Sofia Castro", "edad": 34, "cobertura": "OSDE", "profesional": "Dra. Ibarra", "especialidad": "Dermatologia", "motivo": "dermatitis", "estado": "confirmado", "prioridad": "normal"},
    {"turno": "11:30", "paciente_id": "P008", "paciente": "Hector Lima", "edad": 67, "cobertura": "PAMI", "profesional": "Dr. Salas", "especialidad": "Cardiologia", "motivo": "control post internacion", "estado": "confirmado", "prioridad": "alta"},
    {"turno": "12:00", "paciente_id": "P009", "paciente": "Paula Torres", "edad": 41, "cobertura": "Medife", "profesional": "Dra. Medina", "especialidad": "Clinica medica", "motivo": "receta cronica", "estado": "cancelado", "prioridad": "baja"},
    {"turno": "12:30", "paciente_id": "P010", "paciente": "Diego Martin", "edad": 56, "cobertura": "Omint", "profesional": "Dr. Salas", "especialidad": "Cardiologia", "motivo": "ergometria", "estado": "pendiente", "prioridad": "media"},
]

df = pd.DataFrame(datos)
df["fecha"] = pd.to_datetime(fecha_turnos)
df["alumno"] = ALUMNO  # marcamos el origen

df


,turno,paciente_id,paciente,edad,cobertura,profesional,especialidad,motivo,estado,prioridad,fecha,alumno
0,08:00,P001,Ana Ruiz,45,OSDE,Dra. Medina,Clinica medica,control anual,confirmado,normal,2026-05-23,flavio colavecchia
1,08:30,P002,Carlos Vega,62,PAMI,Dr. Salas,Cardiologia,dolor toracico,confirmado,alta,2026-05-23,flavio colavecchia
2,09:00,P003,Lucia Moreno,38,Swiss Medical,Dra. Medina,Clinica medica,chequeo laboral,confirmado,normal,2026-05-23,flavio colavecchia
3,09:30,P004,Jorge Alvarez,71,PAMI,Dr. Salas,Cardiologia,seguimiento hipertension,confirmado,alta,2026-05-23,flavio colavecchia
4,10:00,P005,Marta Perez,29,Particular,Dra. Ibarra,Dermatologia,control de lunares,pendiente,normal,2026-05-23,flavio colavecchia
5,10:30,P006,Nicolas Ferrer,52,Galeno,Dra. Medina,Clinica medica,laboratorio alterado,confirmado,media,2026-05-23,flavio colavecchia
6,11:00,P007,Sofia Castro,34,OSDE,Dra. Ibarra,Dermatologia,dermatitis,confirmado,normal,2026-05-23,flavio colavecchia
7,11:30,P008,Hector Lima,67,PAMI,Dr. Salas,Cardiologia,control post internacion,confirmado,alta,2026-05-23,flavio colavecchia
8,12:00,P009,Paula Torres,41,Medife,Dra. Medina,Clinica medica,receta cronica,cancelado,baja,2026-05-23,flavio colavecchia
9,12:30,P010,Diego Martin,56,Omint,Dr. Salas,Cardiologia,ergometria,pendiente,media,2026-05-23,flavio colavecchia


## 4. Load #1 — MongoDB Atlas

MongoDB guarda **documentos** (estructuras tipo diccionario de Python). Pandas hace la conversión en una sola línea con `.to_dict('records')`.

In [5]:
from pymongo import MongoClient

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]
collection = db[MONGO_COLLECTION]


In [6]:
print(collection.count_documents({}))

55


In [7]:

# Convertimos el DataFrame a lista de diccionarios
documentos = df.to_dict("records")

# Insertamos
resultado = collection.insert_many(documentos)
print(f"Se insertaron {len(resultado.inserted_ids)} documentos")

Se insertaron 10 documentos


Verificamos leyendo de vuelta lo que acabamos de cargar:


In [ ]:
mis_documentos = list(collection.find({"alumno": ALUMNO}))
print(f"Encontré {len(mis_documentos)} documentos míos en la colección")
for d in mis_documentos[:3]:
    print(d)

Encontré 10 documentos míos en la colección
{'_id': ObjectId('6a36af2234d8458cd817279c'), 'turno': '08:00', 'paciente_id': 'P001', 'paciente': 'Ana Ruiz', 'edad': 45, 'cobertura': 'OSDE', 'profesional': 'Dra. Medina', 'especialidad': 'Clinica medica', 'motivo': 'control anual', 'estado': 'confirmado', 'prioridad': 'normal', 'fecha': datetime.datetime(2026, 5, 23, 0, 0), 'alumno': 'flavio colavecchia'}
{'_id': ObjectId('6a36af2234d8458cd817279d'), 'turno': '08:30', 'paciente_id': 'P002', 'paciente': 'Carlos Vega', 'edad': 62, 'cobertura': 'PAMI', 'profesional': 'Dr. Salas', 'especialidad': 'Cardiologia', 'motivo': 'dolor toracico', 'estado': 'confirmado', 'prioridad': 'alta', 'fecha': datetime.datetime(2026, 5, 23, 0, 0), 'alumno': 'flavio colavecchia'}
{'_id': ObjectId('6a36af2234d8458cd817279e'), 'turno': '09:00', 'paciente_id': 'P003', 'paciente': 'Lucia Moreno', 'edad': 38, 'cobertura': 'Swiss Medical', 'profesional': 'Dra. Medina', 'especialidad': 'Clinica medica', 'motivo': 'chequ

In [9]:
mis_documentos = list(collection.find({"paciente": "Ana Ruiz"}))
print(f"Encontré {len(mis_documentos)} documentos con Ana Ruiz en la colección")
for d in mis_documentos[:3]:
    print(d)

Encontré 7 documentos con Ana Ruiz en la colección
{'_id': ObjectId('6a11b61f786ca7ad12c7db39'), 'turno': '08:00', 'paciente_id': 'P001', 'paciente': 'Ana Ruiz', 'edad': 45, 'cobertura': 'OSDE', 'profesional': 'Dra. Medina', 'especialidad': 'Clinica medica', 'motivo': 'control anual', 'estado': 'confirmado', 'prioridad': 'normal', 'fecha': datetime.datetime(2026, 5, 23, 0, 0), 'alumno': 'flavio'}
{'_id': ObjectId('6a11b667786ca7ad12c7db44'), 'turno': '08:00', 'paciente_id': 'P001', 'paciente': 'Ana Ruiz', 'edad': 45, 'cobertura': 'OSDE', 'profesional': 'Dra. Medina', 'especialidad': 'Clinica medica', 'motivo': 'control anual', 'estado': 'confirmado', 'prioridad': 'normal', 'fecha': datetime.datetime(2026, 5, 23, 0, 0), 'alumno': 'flavio d'}
{'_id': ObjectId('6a3581095a5461aa7a7d16af'), 'turno': '08:00', 'paciente_id': 'P001', 'paciente': 'Ana Ruiz', 'edad': 45, 'cobertura': 'OSDE', 'profesional': 'Dra. Medina', 'especialidad': 'Clinica medica', 'motivo': 'control anual', 'estado': 'con

## 5. Load #2 — Bucket R2 (Parquet)

Acá no hay una base de datos: dejamos un archivo en un bucket de objetos. **Parquet** es un formato columnar comprimido, estándar de facto en data lakes. Mucho más eficiente que CSV para datos tabulares grandes.

In [14]:
import s3fs
import pyarrow as pa
import pyarrow.parquet as pq


# s3fs habla con cualquier almacenamiento S3-compatible (AWS S3, R2, B2, MinIO...)
fs = s3fs.S3FileSystem(
    key=R2_ACCESS_KEY,
    secret=R2_SECRET_KEY,
    client_kwargs={"endpoint_url": R2_ENDPOINT},
)

# Ruta dentro del bucket: una "carpeta" por alumno
ruta_parquet = f"{R2_BUCKET}/{ALUMNO}/consultas_{datetime.now():%Y%m%d_%H%M%S}.parquet"
ruta_parquet 


'diplo-ai/flavio colavecchia/consultas_20260620_122458.parquet'

In [15]:
with fs.open(ruta_parquet, "wb") as f:

    df_parquet = df.copy()
    df_parquet["fecha"] = pd.to_datetime(df_parquet["fecha"]).dt.strftime("%Y-%m-%d")

    table = pa.Table.from_pandas(df_parquet, preserve_index=False)
    pq.write_table(table, f)

print(f"Archivo subido: s3://{ruta_parquet}")


Archivo subido: s3://diplo-ai/flavio colavecchia/consultas_20260620_122458.parquet


Verificamos el archivo que acabamos de subir. En R2 puede pasar que una credencial permita leer/escribir objetos pero no listar carpetas, así que primero chequeamos la ruta exacta.


In [16]:
print(f"Ruta del Parquet: s3://{ruta_parquet}")

try:
    existe = fs.exists(ruta_parquet)
    print(f"¿Existe el objeto? {existe}")
except Exception as e:
    print("No pude verificar la existencia con HEAD/exists.")
    print(type(e).__name__, str(e))

try:
    archivos = fs.ls(f"{R2_BUCKET}/{ALUMNO}/")
    print()
    print("Archivos en mi carpeta del bucket:")
    for a in archivos:
        print(" -", a)
except FileNotFoundError:
    archivos = [ruta_parquet]
    print()
    print("No pude listar la carpeta del bucket.")
    print("Esto no implica que el archivo no exista: puede ser un permiso de listado del bucket.")
    print("Seguimos con la ruta exacta del Parquet:")
    print(" -", ruta_parquet)


Ruta del Parquet: s3://diplo-ai/flavio colavecchia/consultas_20260620_122458.parquet
¿Existe el objeto? True

No pude listar la carpeta del bucket.
Esto no implica que el archivo no exista: puede ser un permiso de listado del bucket.
Seguimos con la ruta exacta del Parquet:
 - diplo-ai/flavio colavecchia/consultas_20260620_122458.parquet


Y leemos el Parquet usando la ruta exacta del objeto, sin depender del listado del bucket:


In [17]:
with fs.open(ruta_parquet, "rb") as f:
    # Evita el error de extensión `pandas.period` en algunos entornos pyarrow/pandas
    table_leida = pq.read_table(f, use_pandas_metadata=False)
    df_leido = table_leida.to_pandas()

df_leido


,turno,paciente_id,paciente,edad,cobertura,profesional,especialidad,motivo,estado,prioridad,fecha,alumno
0,08:00,P001,Ana Ruiz,45,OSDE,Dra. Medina,Clinica medica,control anual,confirmado,normal,2026-05-23,flavio colavecchia
1,08:30,P002,Carlos Vega,62,PAMI,Dr. Salas,Cardiologia,dolor toracico,confirmado,alta,2026-05-23,flavio colavecchia
2,09:00,P003,Lucia Moreno,38,Swiss Medical,Dra. Medina,Clinica medica,chequeo laboral,confirmado,normal,2026-05-23,flavio colavecchia
3,09:30,P004,Jorge Alvarez,71,PAMI,Dr. Salas,Cardiologia,seguimiento hipertension,confirmado,alta,2026-05-23,flavio colavecchia
4,10:00,P005,Marta Perez,29,Particular,Dra. Ibarra,Dermatologia,control de lunares,pendiente,normal,2026-05-23,flavio colavecchia
5,10:30,P006,Nicolas Ferrer,52,Galeno,Dra. Medina,Clinica medica,laboratorio alterado,confirmado,media,2026-05-23,flavio colavecchia
6,11:00,P007,Sofia Castro,34,OSDE,Dra. Ibarra,Dermatologia,dermatitis,confirmado,normal,2026-05-23,flavio colavecchia
7,11:30,P008,Hector Lima,67,PAMI,Dr. Salas,Cardiologia,control post internacion,confirmado,alta,2026-05-23,flavio colavecchia
8,12:00,P009,Paula Torres,41,Medife,Dra. Medina,Clinica medica,receta cronica,cancelado,baja,2026-05-23,flavio colavecchia
9,12:30,P010,Diego Martin,56,Omint,Dr. Salas,Cardiologia,ergometria,pendiente,media,2026-05-23,flavio colavecchia


## 6. Load #3 — Bucket R2 (PDF)

Ahora generamos un **reporte operativo** con la lista de pacientes del día. A diferencia del Parquet, este archivo no está pensado para análisis masivo, sino para circularlo, imprimirlo o archivarlo como documento.


In [18]:
from pathlib import Path

reportes_dir = Path("data/reportes")
reportes_dir.mkdir(parents=True, exist_ok=True)

fecha_reporte = pd.to_datetime(fecha_turnos).strftime("%d/%m/%Y")
pdf_local = reportes_dir / f"pacientes_{ALUMNO}_{datetime.now():%Y%m%d_%H%M%S}.pdf"

confirmados = int((df["estado"] == "confirmado").sum())
pendientes = int((df["estado"] == "pendiente").sum())
altas = int((df["prioridad"] == "alta").sum())


def ajustar(texto, ancho):
    texto = str(texto)
    return texto[: ancho - 1] + "..." if len(texto) > ancho else texto.ljust(ancho)


def escapar_pdf(texto):
    return str(texto).replace("\\", "\\\\").replace("(", "\\(").replace(")", "\\)")


columnas = [
    ("turno", 5),
    ("paciente_id", 5),
    ("paciente", 18),
    ("edad", 4),
    ("cobertura", 14),
    ("profesional", 15),
    ("especialidad", 14),
    ("motivo", 24),
    ("estado", 10),
    ("prioridad", 8),
]

lineas = [
    "LISTA DE PACIENTES DEL DIA",
    f"Fecha: {fecha_reporte} | Total: {len(df)} | Confirmados: {confirmados} | Pendientes: {pendientes} | Prioridad alta: {altas}",
    "",
]

encabezado = " | ".join(ajustar(nombre, ancho) for nombre, ancho in columnas)
lineas.append(encabezado)
lineas.append("-" * len(encabezado))

for _, fila in df.iterrows():
    lineas.append(" | ".join(ajustar(fila[nombre], ancho) for nombre, ancho in columnas))

# PDF mínimo en una página apaisada A4. Usamos fuente Courier para alinear la tabla.
comandos = ["BT", "/F1 15 Tf", "40 550 Td", f"({escapar_pdf(lineas[0])}) Tj", "/F1 8 Tf", "0 -18 Td"]
for linea in lineas[1:]:
    comandos.append(f"({escapar_pdf(linea)}) Tj")
    comandos.append("0 -11 Td")
comandos.append("ET")
stream = "\n".join(comandos).encode("latin-1", errors="replace")

objetos = [
    b"1 0 obj\n<< /Type /Catalog /Pages 2 0 R >>\nendobj\n",
    b"2 0 obj\n<< /Type /Pages /Kids [3 0 R] /Count 1 >>\nendobj\n",
    b"3 0 obj\n<< /Type /Page /Parent 2 0 R /MediaBox [0 0 842 595] /Resources << /Font << /F1 4 0 R >> >> /Contents 5 0 R >>\nendobj\n",
    b"4 0 obj\n<< /Type /Font /Subtype /Type1 /BaseFont /Courier >>\nendobj\n",
    b"5 0 obj\n<< /Length " + str(len(stream)).encode("ascii") + b" >>\nstream\n" + stream + b"\nendstream\nendobj\n",
]

pdf = b"%PDF-1.4\n%\xe2\xe3\xcf\xd3\n"
offsets = [0]
for obj in objetos:
    offsets.append(len(pdf))
    pdf += obj

xref_pos = len(pdf)
pdf += f"xref\n0 {len(objetos) + 1}\n".encode("ascii")
pdf += b"0000000000 65535 f \n"
for offset in offsets[1:]:
    pdf += f"{offset:010d} 00000 n \n".encode("ascii")
pdf += f"trailer\n<< /Size {len(objetos) + 1} /Root 1 0 R >>\nstartxref\n{xref_pos}\n%%EOF\n".encode("ascii")

pdf_local.write_bytes(pdf)
print(f"PDF generado: {pdf_local}")


PDF generado: data/reportes/pacientes_flavio colavecchia_20260620_123247.pdf


Subimos el PDF al mismo bucket, pero en una carpeta `reportes/` para separarlo de los datos analíticos.


In [19]:
ruta_pdf = f"{R2_BUCKET}/{ALUMNO}/reportes/{pdf_local.name}"

with fs.open(ruta_pdf, "wb") as destino:
    destino.write(pdf_local.read_bytes())

print(f"PDF subido: s3://{ruta_pdf}")


PDF subido: s3://diplo-ai/flavio colavecchia/reportes/pacientes_flavio colavecchia_20260620_123247.pdf


Verificamos listando solamente la carpeta de reportes:


In [20]:
print(f"Ruta del PDF: s3://{ruta_pdf}")

try:
    existe = fs.exists(ruta_pdf)
    print(f"¿Existe el objeto? {existe}")
except Exception as e:
    print("No pude verificar la existencia con HEAD/exists.")
    print(type(e).__name__, str(e))

try:
    reportes = fs.ls(f"{R2_BUCKET}/{ALUMNO}/reportes/")
    print()
    print("Reportes PDF en el bucket:")
    for r in reportes:
        print(" -", r)
except FileNotFoundError:
    reportes = [ruta_pdf]
    print()
    print("No pude listar la carpeta de reportes del bucket.")
    print("Esto no implica que el PDF no exista: puede ser un permiso de listado del bucket.")
    print("El PDF subido queda en:")
    print(" -", ruta_pdf)


Ruta del PDF: s3://diplo-ai/flavio colavecchia/reportes/pacientes_flavio colavecchia_20260620_123247.pdf
¿Existe el objeto? True

No pude listar la carpeta de reportes del bucket.
Esto no implica que el PDF no exista: puede ser un permiso de listado del bucket.
El PDF subido queda en:
 - diplo-ai/flavio colavecchia/reportes/pacientes_flavio colavecchia_20260620_123247.pdf


## 7. Reflexión

Mismos datos, tres destinos muy distintos:

| | MongoDB | Bucket Parquet | Bucket PDF |
|---|---|---|---|
| **Forma del dato** | Documento (dict) | Archivo columnar | Documento final |
| **Lectura típica** | Por filtros, registro a registro | Por columnas, en batch | Humana, visual, compartible |
| **Cuándo conviene** | Apps que consultan registros vivos | Analítica, ML, históricos | Reportes, auditoría, comunicación |
| **Modificación** | Update sobre un documento | Reescribir el archivo | Regenerar el documento |

En un pipeline real es muy común tener **varios loads** para el mismo dataset: Mongo (o Postgres) para la operación del producto, un data lake para análisis, y documentos o adjuntos para procesos administrativos. La etapa de Load decide cuál es cuál.
